# So sánh 3 model: XGBoost (hiện tại) / TabM / LSTM cho chiến lược H1 touch + Direction

**Giữ giống nhau cho cả 3 model (chỉ đổi model học):**
- **Dữ liệu và feature:** touch 92 feature lean; Direction 37 feature (có hướng + vị thế).
- **Nhãn:** touch ±b trong 8h; Direction 3 lớp.
- **Walk-forward theo quý:** dev 2022–2024, test 2025–2026. Train tới Q − 2 quý; 2 quý dùng để dừng sớm và hiệu chỉnh.
- **Ensemble 5 seed.**
- **Touch:** trọng số thời gian (half-life 540 ngày) và hiệu chỉnh nhiệt độ.
- **Ngưỡng trượt và luật lệnh:** touch 14 ngày, Direction 90 ngày; TP = SL = 1b, giữ ≤ 9h.

**Model** (`model_training/nn_models.py`):
| model | đầu vào | kiến trúc |
|---|---|---|
| XGBoost | feature nến t | cây (cấu hình production) |
| TabM | feature nến t | MLP 3 × 256 + BatchEnsemble k = 16 (Gorishniy và cộng sự, ICLR 2025) |
| LSTM | chuỗi feature 24 nến gần nhất (t−23 … t) | Linear 64 → LSTM 2 lớp × 64 → Linear |

Không tinh chỉnh siêu tham số của TabM / LSTM trên test. Muốn đổi thì sửa `nn_params` và chạy lại; cache theo tham số.

**So sánh:**
1. AUC touch và AUC Direction theo quý.
2. Giao dịch với **cùng tham số đã khoá** của XGBoost.
3. Mỗi model **tự chọn tham số trên dev**, rồi đo test.
4. Ghép chéo: touch của model A + Direction của model B.
5. Paper futures cross x50, margin 1% vốn.

**Tiêu chí:** TB R/lệnh của 5 seed tăng VÀ dải seed hẹp lại (seed tệ nhất tăng).

**Thời gian:** XGBoost lấy từ cache. TabM và LSTM phải train 20 quý × 5 seed × 2 model (touch + Direction). Trên GPU (torch cu118 trong .venv) mất cỡ vài chục phút đến vài giờ. Trên CPU lâu hơn nhiều. Mỗi quý × seed được cache ở `artifacts/1h_strategy_v2/folds/{tabm|lstm}_*`, nên dừng giữa chừng rồi chạy lại vẫn tiếp tục được.

In [1]:
import os, sys
from dataclasses import replace
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "model_training").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "model_training"))
import numpy as np
import pandas as pd
import torch
import strategy_h1 as st
import futures_paper as fu

pd.set_option("display.width", 240, "display.max_columns", 50)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
cfg = st.load_locked("artifacts/1h_strategy_v2/config_locked.json")
OUT = Path(cfg.output_dir) / "compare_models"
OUT.mkdir(parents=True, exist_ok=True)
D = st.build(cfg)
MODELS = {"XGBoost": cfg,
          "TabM": replace(cfg, model_kind="tabm", nn_params={}),     # mặc định nn_models.TabMParams
          "LSTM": replace(cfg, model_kind="lstm", nn_params={})}     # mặc định nn_models.LSTMParams
P, REP = {}, {}

torch 2.5.1+cu118 | cuda: True NVIDIA GeForce RTX 2050
Dữ liệu: 54,979 nến 1h, 53,522 mẫu (2020-08-30 -> 2026-10-08), touch 92 feature, direction 37 feature, 17s


## 1. Walk-forward (mỗi model một ô, chạy riêng được; kết quả cache theo quý × seed)

In [2]:
name = "XGBoost"
P[name] = {ph: st.walk_forward(D, MODELS[name], ph) for ph in ("dev", "test")}

dev 2022Q1: AUC touch 0.610 | AUC direction 0.616 | train 7,303 | 0s
dev 2022Q2: AUC touch 0.695 | AUC direction 0.536 | train 9,511 | 0s
dev 2022Q3: AUC touch 0.605 | AUC direction 0.520 | train 11,719 | 0s
dev 2022Q4: AUC touch 0.689 | AUC direction 0.601 | train 13,879 | 0s
dev 2023Q1: AUC touch 0.664 | AUC direction 0.675 | train 16,063 | 0s
dev 2023Q2: AUC touch 0.692 | AUC direction 0.588 | train 18,271 | 0s
dev 2023Q3: AUC touch 0.751 | AUC direction 0.565 | train 20,479 | 0s
dev 2023Q4: AUC touch 0.721 | AUC direction 0.566 | train 22,639 | 0s
dev 2024Q1: AUC touch 0.718 | AUC direction 0.571 | train 24,823 | 0s
dev 2024Q2: AUC touch 0.749 | AUC direction 0.587 | train 27,031 | 0s
dev 2024Q3: AUC touch 0.735 | AUC direction 0.585 | train 29,239 | 0s
dev 2024Q4: AUC touch 0.778 | AUC direction 0.626 | train 31,423 | 0s
test 2025Q1: AUC touch 0.763 | AUC direction 0.552 | train 33,607 | 0s
test 2025Q2: AUC touch 0.729 | AUC direction 0.556 | train 35,815 | 0s
test 2025Q3: AUC tou

In [3]:
name = "TabM"
P[name] = {ph: st.walk_forward(D, MODELS[name], ph) for ph in ("dev", "test")}

dev 2022Q1: AUC touch 0.624 | AUC direction 0.586 | train 7,303 | 66s
dev 2022Q2: AUC touch 0.675 | AUC direction 0.561 | train 9,511 | 64s
dev 2022Q3: AUC touch 0.597 | AUC direction 0.527 | train 11,719 | 91s
dev 2022Q4: AUC touch 0.642 | AUC direction 0.633 | train 13,879 | 90s
dev 2023Q1: AUC touch 0.613 | AUC direction 0.677 | train 16,063 | 86s
dev 2023Q2: AUC touch 0.668 | AUC direction 0.586 | train 18,271 | 107s
dev 2023Q3: AUC touch 0.737 | AUC direction 0.551 | train 20,479 | 136s
dev 2023Q4: AUC touch 0.719 | AUC direction 0.582 | train 22,639 | 149s
dev 2024Q1: AUC touch 0.711 | AUC direction 0.591 | train 24,823 | 155s
dev 2024Q2: AUC touch 0.745 | AUC direction 0.612 | train 27,031 | 195s
dev 2024Q3: AUC touch 0.719 | AUC direction 0.588 | train 29,239 | 179s
dev 2024Q4: AUC touch 0.780 | AUC direction 0.619 | train 31,423 | 190s
test 2025Q1: AUC touch 0.759 | AUC direction 0.567 | train 33,607 | 190s
test 2025Q2: AUC touch 0.711 | AUC direction 0.548 | train 35,815 | 23

In [4]:
name = "LSTM"
P[name] = {ph: st.walk_forward(D, MODELS[name], ph) for ph in ("dev", "test")}

dev 2022Q1: AUC touch 0.582 | AUC direction 0.621 | train 7,303 | 36s
dev 2022Q2: AUC touch 0.669 | AUC direction 0.545 | train 9,511 | 38s
dev 2022Q3: AUC touch 0.610 | AUC direction 0.525 | train 11,719 | 42s
dev 2022Q4: AUC touch 0.660 | AUC direction 0.655 | train 13,879 | 50s
dev 2023Q1: AUC touch 0.639 | AUC direction 0.652 | train 16,063 | 50s
dev 2023Q2: AUC touch 0.655 | AUC direction 0.633 | train 18,271 | 59s
dev 2023Q3: AUC touch 0.708 | AUC direction 0.530 | train 20,479 | 60s
dev 2023Q4: AUC touch 0.696 | AUC direction 0.599 | train 22,639 | 62s
dev 2024Q1: AUC touch 0.699 | AUC direction 0.577 | train 24,823 | 68s
dev 2024Q2: AUC touch 0.727 | AUC direction 0.593 | train 27,031 | 78s
dev 2024Q3: AUC touch 0.712 | AUC direction 0.586 | train 29,239 | 80s
dev 2024Q4: AUC touch 0.761 | AUC direction 0.587 | train 31,423 | 84s
test 2025Q1: AUC touch 0.742 | AUC direction 0.560 | train 33,607 | 89s
test 2025Q2: AUC touch 0.706 | AUC direction 0.538 | train 35,815 | 95s
test 2

## 2. AUC theo quý

In [5]:
R = pd.concat([P[m][ph][1].assign(model=m, phase=ph) for m in P for ph in ("dev", "test")], ignore_index=True)
R.to_csv(OUT / "auc_by_quarter.csv", index=False)
cols = ["AUC touch", "AUC direction (đã chạm)", "AUC touch của Direction"]
display(R.groupby(["phase", "model"], sort=False)[cols].agg(["mean", "min"]).round(4))
R.pivot_table(index="quý", columns="model", values=["AUC touch", "AUC direction (đã chạm)"]).round(3)

AUC touch         AUC direction (đã chạm)         AUC touch của Direction        
                   mean     min                    mean     min                    mean     min
phase model                                                                                    
dev   XGBoost    0.7007  0.6053                  0.5862  0.5195                  0.5956  0.5372
test  XGBoost    0.7371  0.7132                  0.5696  0.5304                  0.6417  0.6044
dev   TabM       0.6859  0.5967                  0.5927  0.5272                  0.5956  0.5131
test  TabM       0.7338  0.6991                  0.5768  0.5191                  0.6354  0.6087
dev   LSTM       0.6763  0.5818                  0.5920  0.5249                  0.5802  0.4896
test  LSTM       0.7244  0.7044                  0.5625  0.4891                  0.6172  0.5832

AUC direction (đã chạm)                AUC touch               
model                     LSTM   TabM XGBoost      LSTM   TabM XGBoost
quý                                                                   
2022Q1                   0.621  0.586   0.616     0.582  0.624   0.610
2022Q2                   0.545  0.561   0.536     0.669  0.675   0.695
2022Q3                   0.525  0.527   0.520     0.610  0.597   0.605
2022Q4                   0.655  0.633   0.601     0.660  0.642   0.689
2023Q1                   0.652  0.677   0.675     0.639  0.613   0.664
2023Q2                   0.633  0.586   0.588     0.655  0.668   0.692
2023Q3                   0.530  0.551   0.565     0.708  0.737   0.751
2023Q4                   0.599  0.582   0.566     0.696  0.719   0.721
2024Q1                   0.577  0.591   0.571     0.699  0.711   0.718
2024Q2                   0.593  0.612   0.587     0.727  0.745   0.749
2024Q3                   0.586  0.588   0.585     0.712  0.719   0.735
2024Q4                   0.587  0.619   0.626     0.761  0.780   0.778
2025Q1                   0.560  0.567   0.552     0.742  0.759   0.763
2025Q2                   0.538  0.548   0.556     0.706  0.711   0.729
2025Q3                   0.489  0.519   0.563     0.704  0.699   0.713
2025Q4                   0.524  0.547   0.530     0.727  0.758   0.758
2026Q1                   0.557  0.572   0.570     0.739  0.747   0.743
2026Q2                   0.592  0.599   0.591     0.732  0.736   0.741
2026Q3                   0.535  0.572   0.564     0.705  0.715   0.725
2026Q4                   0.705  0.690   0.631     0.740  0.745   0.726

## 3. Giao dịch với cùng tham số đã khoá (của XGBoost)

In [6]:
def trade_row(name, c, Pd, Pt):
    dev, test = st.evaluate(D, Pd, c, "dev"), st.evaluate(D, Pt, c, "test")
    S = st.seed_table(D, Pt, c, "test")
    s, sd = test["summary"], dev["summary"]
    out = {"model": name, "touch_top": c.touch_top, "dir_top": c.dir_top, "max_pos": c.max_positions, "gap": c.min_gap_bars,
           "vào": c.entry_mode, "dev lệnh/năm": np.mean(list(sd["trades_by_year"].values())), "dev R/lệnh": sd["R_per_trade"],
           "test lệnh": s["trades"], "test R ens": s["R_per_trade"],
           **{f"R {y}": s["R_by_year"][y] / s["trades_by_year"][y] for y in sorted(s["trades_by_year"])},
           "TB seed": S.loc["TB seed", "R/lệnh"], "std seed": S.loc["std seed", "R/lệnh"],
           "seed tệ nhất": S.loc["seed tệ nhất", "R/lệnh"], "thắng": s["win_rate"], "hết giờ": s["timeout_share"],
           "DD % vốn": s["max_dd_pct"], "lãi % vốn": (s["equity_multiple"] - 1) * 100, "% giá/lệnh": s["pct_per_trade"],
           "quý âm": s["negative_quarters"], "LONG": s["long_share"]}
    return out, test, S

LOCK = {}
rows = []
for m in P:
    r, LOCK[m], S = trade_row(m, cfg, P[m]["dev"][0], P[m]["test"][0])
    rows.append(r)
    S.to_csv(OUT / f"seeds_locked_{m}.csv")
A = pd.DataFrame(rows).set_index("model")
A.to_csv(OUT / "trade_locked_params.csv")
A.round(3)

,touch_top,dir_top,max_pos,gap,vào,dev lệnh/năm,dev R/lệnh,test lệnh,test R ens,R 2025,R 2026,TB seed,std seed,seed tệ nhất,thắng,hết giờ,DD % vốn,lãi % vốn,% giá/lệnh,quý âm,LONG
model,,,,,,,,,,,,,,,,,,,,,
XGBoost,0.2,0.1,5,1,limit_pullback,201.000,0.178,400,0.168,0.150,0.190,0.155,0.025,0.133,0.585,0.242,11.111,87.978,0.213,1,0.500
TabM,0.2,0.1,5,1,limit_pullback,197.667,0.178,352,0.107,0.056,0.161,0.090,0.024,0.063,0.548,0.253,12.823,40.684,0.153,2,0.562
LSTM,0.2,0.1,5,1,limit_pullback,204.667,0.150,352,0.188,0.119,0.253,0.127,0.053,0.069,0.585,0.261,17.519,86.709,0.260,1,0.537


In [7]:
# R/lệnh test theo quý
pd.DataFrame({m: LOCK[m]["by_quarter"]["R/lệnh"] for m in LOCK}).round(3)

,XGBoost,TabM,LSTM
quý,,,
2025Q1,0.057,0.040,0.178
2025Q2,0.324,0.216,0.471
2025Q3,-0.001,-0.229,-0.418
2025Q4,0.211,0.131,0.035
2026Q1,0.188,0.181,0.197
2026Q2,0.109,0.193,0.315
2026Q3,0.381,0.159,0.198
2026Q4,0.930,-1.065,0.936


## 4. Mỗi model tự chọn tham số trên dev (cùng lưới, ≥ min_trades_per_year lệnh/năm), đo test

In [8]:
SEL, rows = {}, []
for m in P:
    base = replace(MODELS[m], locked_from="")
    G = st.grid_search(D, P[m]["dev"][0], base)
    G.to_csv(OUT / f"grid_dev_{m}.csv", index=False)
    c = st.lock_config(base, st.select_best(G), G, path=str(OUT / f"config_locked_{m}.json"))
    r, SEL[m], S = trade_row(m, c, P[m]["dev"][0], P[m]["test"][0])
    rows.append(r)
    S.to_csv(OUT / f"seeds_reselected_{m}.csv")
    display(st.nested_selection(G, base).assign(model=m))
B = pd.DataFrame(rows).set_index("model")
B.to_csv(OUT / "trade_reselected.csv")
B.round(3)

Lưới 96 ô, 76 ô hợp lệ (>= 150 lệnh/năm), 20s


,năm test,chọn bằng,touch_top,dir_top,max_positions,min_gap_bars,entry_mode,R/lệnh năm test,lệnh năm test,TB toàn lưới năm test,model
0,2023,2022,0.5,0.05,5,2,next_open,0.245735,176,0.192142,XGBoost
1,2024,2022-2023,0.5,0.05,5,2,next_open,0.090690,184,0.118541,XGBoost


Lưới 96 ô, 72 ô hợp lệ (>= 150 lệnh/năm), 19s


,năm test,chọn bằng,touch_top,dir_top,max_positions,min_gap_bars,entry_mode,R/lệnh năm test,lệnh năm test,TB toàn lưới năm test,model
0,2023,2022,0.5,0.05,5,1,next_open,0.237058,245,0.184717,TabM
1,2024,2022-2023,0.5,0.05,5,1,next_open,0.221440,221,0.176566,TabM


Lưới 96 ô, 68 ô hợp lệ (>= 150 lệnh/năm), 17s


,năm test,chọn bằng,touch_top,dir_top,max_positions,min_gap_bars,entry_mode,R/lệnh năm test,lệnh năm test,TB toàn lưới năm test,model
0,2023,2022,0.2,0.10,5,2,next_open,0.136877,125,0.152704,LSTM
1,2024,2022-2023,0.3,0.05,5,1,next_open,0.124403,139,0.108543,LSTM


,touch_top,dir_top,max_pos,gap,vào,dev lệnh/năm,dev R/lệnh,test lệnh,test R ens,R 2025,R 2026,TB seed,std seed,seed tệ nhất,thắng,hết giờ,DD % vốn,lãi % vốn,% giá/lệnh,quý âm,LONG
model,,,,,,,,,,,,,,,,,,,,,
XGBoost,0.2,0.10,5,1,limit_pullback,201.000,0.178,400,0.168,0.150,0.190,0.155,0.025,0.133,0.585,0.242,11.111,87.978,0.213,1,0.500
TabM,0.5,0.05,5,1,next_open,225.667,0.194,449,-0.011,-0.045,0.029,0.018,0.032,-0.018,0.470,0.345,29.947,-8.658,-0.022,4,0.539
LSTM,0.3,0.10,5,2,limit_pullback,182.000,0.205,318,0.102,0.019,0.187,0.112,0.044,0.038,0.544,0.292,16.622,35.166,0.138,3,0.541


## 5. Ghép chéo: touch của model A + Direction của model B (tham số đã khoá, test)

In [9]:
def combine(Pt_touch, Pt_dir):
    assert np.array_equal(Pt_touch.t_ns.to_numpy(), Pt_dir.t_ns.to_numpy())
    Q = Pt_touch.copy()
    for c in Pt_dir.columns:
        if c.startswith("p_up") or c.startswith("p_dtouch"):
            Q[c] = Pt_dir[c].to_numpy()
    return Q

rows = []
for mt in P:
    for md_ in P:
        Qd, Qt = combine(P[mt]["dev"][0], P[md_]["dev"][0]), combine(P[mt]["test"][0], P[md_]["test"][0])
        r, _, _ = trade_row(f"touch {mt} + Direction {md_}", cfg, Qd, Qt)
        rows.append({"touch": mt, "Direction": md_, **{k: r[k] for k in ("dev R/lệnh", "test lệnh", "test R ens", "TB seed",
                                                                          "std seed", "seed tệ nhất", "DD % vốn", "lãi % vốn")}})
X = pd.DataFrame(rows)
X.to_csv(OUT / "cross_touch_direction.csv", index=False)
display(X.round(3))
X.pivot(index="touch", columns="Direction", values="TB seed").round(3)

,touch,Direction,dev R/lệnh,test lệnh,test R ens,TB seed,std seed,seed tệ nhất,DD % vốn,lãi % vốn
0,XGBoost,XGBoost,0.178,400,0.168,0.155,0.025,0.133,11.111,87.978
1,XGBoost,TabM,0.183,363,0.154,0.122,0.027,0.092,9.415,68.615
2,XGBoost,LSTM,0.186,376,0.193,0.180,0.059,0.112,14.788,97.376
3,TabM,XGBoost,0.176,400,0.152,0.137,0.030,0.100,10.185,77.324
4,TabM,TabM,0.178,352,0.107,0.090,0.024,0.063,12.823,40.684
5,TabM,LSTM,0.156,379,0.203,0.150,0.056,0.082,10.878,107.663
6,LSTM,XGBoost,0.154,378,0.107,0.107,0.022,0.073,12.334,44.810
7,LSTM,TabM,0.169,345,0.085,0.092,0.022,0.066,16.501,29.810
8,LSTM,LSTM,0.150,352,0.188,0.127,0.053,0.069,17.519,86.709


Direction,LSTM,TabM,XGBoost
touch,,,
LSTM,0.127,0.092,0.107
TabM,0.150,0.090,0.137
XGBoost,0.180,0.122,0.155


## 6. Paper futures cross x50, margin 1% vốn, vốn 100,000 USDT (test)

In [10]:
funding = pd.read_csv(cfg.funding_csv)
fp = fu.FuturesParams(capital=100_000, leverage=50, margin_frac=0.01, compound=True, margin_mode="cross")
rows = []
for m in P:
    for how, res_, c in (("khoá chung", LOCK[m], cfg), ("tự chọn trên dev", SEL[m], st.load_locked(str(OUT / f"config_locked_{m}.json")))):
        sig = res_["signals"][res_["signals"].signal]
        r = fu.run(D, sig, c, fp, funding)
        y = fu.by_period(r, "Y")["lãi % vốn"]
        s = r["summary"]
        rows.append({"model": m, "tham số": how, "lệnh": s["lệnh"], "vốn cuối": s["vốn cuối"], "lãi %": s["lãi %"],
                     "CAGR %": s["lãi %/năm (CAGR)"], "DD %": s["DD max %"], "thắng": s["thắng"],
                     "TB % vốn/lệnh": s["TB lãi/lệnh % vốn"], "tổng phí": s["tổng phí"],
                     **{f"lãi % {k}": v for k, v in y.items()}})
F = pd.DataFrame(rows)
F.to_csv(OUT / "futures_x50.csv", index=False)
F.round(2)

,model,tham số,lệnh,vốn cuối,lãi %,CAGR %,DD %,thắng,TB % vốn/lệnh,tổng phí,lãi % 2025,lãi % 2026
0,XGBoost,khoá chung,400,155181.31,55.18,28.68,7.42,0.58,0.11,15482.02,20.67,28.60
1,XGBoost,tự chọn trên dev,400,155181.31,55.18,28.68,7.42,0.58,0.11,15482.02,20.67,28.60
2,TabM,khoá chung,352,132657.22,32.66,17.73,6.49,0.55,0.08,12873.77,6.27,24.83
3,TabM,tự chọn trên dev,449,96332.76,-3.67,-2.13,15.63,0.47,-0.00,19430.96,-6.59,3.13
4,LSTM,khoá chung,352,160265.14,60.27,31.31,9.86,0.59,0.14,14082.03,14.58,39.88
5,LSTM,tự chọn trên dev,318,126305.84,26.31,14.44,8.96,0.54,0.08,11533.33,1.59,24.33
